In [ ]:
# 0. Erst die Bibliotheken installieren (falls noch nicht geschehen)
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl

In [ ]:
import torch
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

# --- 1. Basic Configuration ---
model_id = "unsloth/llama-3-8b-instruct-bnb-4bit"
output_dir = "./llama3-tax-standard"

# --- 2. Load Resources Independently ---
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# --- 3. PEFT Setup (LoRA) ---
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# --- 4. Robust Tokenization Function ---
# Instead of relying on the Trainer to format, we tokenize the data ourselves.
# This avoids all "dataset_text_field" or "max_seq_length" TypeErrors.
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=1024 # Conservative length to save memory
    )

# Load the master CSV you prepared
raw_dataset = load_dataset("csv", data_files="train_dataset_final.csv", split="train")

# Map the tokenization across the dataset
tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# --- 5. Standard Training Pipeline ---
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_steps=60,
    logging_steps=1,
    fp16=True,
    optim="paged_adamw_32bit",
    report_to="none",
    remove_unused_columns=False
)

# Use the standard Trainer with a Data Collator
# This is the most stable combination in the Hugging Face ecosystem
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Starting standard training pipeline...")
trainer.train()
print("Training completed successfully.")

In [ ]:
import pandas as pd
import torch

# --- 1. Inference Optimization ---
# Set the model to evaluation mode to disable dropout and other training-specific behaviors
model.eval()

# Configure tokenizer for batch inference:
# Left-padding is mandatory for causal LLMs like Llama-3 during batch generation
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 2. Data Loading ---
# Load the cleaned test dataset containing the legal cases
df_test = pd.read_csv("dataset_clean.csv")
results = []

# --- 3. High-Performance Batch Inference ---
# Batching increases GPU utilization and significantly reduces total execution time
BATCH_SIZE = 16
print(f"Starting optimized batch inference for {len(df_test)} cases (Batch Size: {BATCH_SIZE})...")

for i in range(0, len(df_test), BATCH_SIZE):
    # Extract the current batch of data
    batch_df = df_test.iloc[i : i + BATCH_SIZE]

    # Construct prompts using the fine-tuned instruction format
    batch_prompts = [
        f"### Instruction:\nAnalysiere den folgenden steuerrechtlichen Sachverhalt und nenne die relevanten österreichischen Paragraphen.\n\n### Input:\n{row['prompt']}\n\n### Response:\n"
        for _, row in batch_df.iterrows()
    ]

    # Tokenize the batch with padding and truncation to ensure uniform tensor shapes
    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048
    ).to("cuda")

    # Execute inference using torch.inference_mode for reduced memory overhead and faster computation
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,      # Constrain generation length as legal citations are concise
            use_cache=True,          # Enable KV-caching for faster decoding
            do_sample=False,         # Use greedy decoding for deterministic and reproducible legal output
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode batch outputs and remove special tokens
    decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Parse generated responses and map them back to their original IDs
    for idx, full_text in enumerate(decoded_outputs):
        # Extract content following the '### Response:' marker
        if "### Response:\n" in full_text:
            answer = full_text.split("### Response:\n")[-1].strip()
        else:
            answer = full_text.strip()

        results.append({
            "id": batch_df.iloc[idx]['id'],
            "answer": answer
        })

    # Console feedback for monitoring processing status
    current_progress = min(i + BATCH_SIZE, len(df_test))
    completion_rate = (current_progress / len(df_test)) * 100
    print(f"Progress: {current_progress}/643 ({completion_rate:.1f}%)")

# --- 4. Result Export ---
# Save the model predictions to a CSV file for final evaluation
output_df = pd.DataFrame(results)
output_df.to_csv("model_2_results_final.csv", index=False)
print("\nInference completed. Results exported to 'model_2_results_final.csv'.")